# Day 26 — Webhooks and API Keys: HMAC Signatures, Idempotency & Background Tasks (19 Nov)

*Rebuilt from the master Google Doc session (lesson text and every code example). Code cells are not executed: several need Postgres, a running server or files from the original starter zip.*

## Today at a glance

- Goal: let Contentstack push changes into your service safely. By tonight you can receive a webhook in FastAPI, prove it came from the CMS with an HMAC signature, process each event exactly once even when it is delivered twice or out of order, protect a read endpoint with an API key, and move slow follow-up work into a background task.

- Deliverable: in day26/, Acme's content sync service. You write signatures.py (3 functions), api_keys.py (3 functions), repository.py (4 pieces), the body of the webhook route in sync_api.py, and 3 tests in test_mine.py. Done means python3 -m pytest -q shows 47 passed (44 given cases plus your 3), plus delivery_log.txt from a manual run, and DSA in dsa.py.

- Where this sits: Week 4, Phase 1 (Foundation), day 5 of 7. Day 24 gave you the layers, Day 25 the test fixtures; today adds the two ways other systems talk to your API: they call you (webhooks) and they authenticate (API keys). Day 27 packages Phase 1 as one project, and Project 2 (Week 11, an agent on a headless CMS) starts from this receiver.

- Honest check: the repo still shows nothing after Day 4, and Phase 1 ends in two days. A webhook receiver with signature checks and 47 green tests is a strong interview story for FDE roles, but only if a hiring manager can open it. Push today, even half done.

Time split (5h): the Week 4 split from the Day 21 plan.

| Block | Time | What you do |
|---|---|---|
| Theory | 1.5h | Five concepts below, typing every example; check-your-understanding predictions |
| Core task | 2h | signatures.py (30 min), api_keys.py (15 min), repository.py (30 min), the route (30 min), test_mine.py and the manual run (15 min) |
| Catch-up | 1h | Push Days 22 to 25 as they are (15 min); then Day 25 until its 37 tests pass |
| DSA | 0.5h | constant_time_equals and apply_events in dsa.py |

Setup (15 min): a fresh virtual environment inside day26/ and the same packages as Day 25. The server uses your fde database; the tests use fde_test and never touch fde. If your Postgres needs a user and password, set ACME_TEST_DATABASE_URL as on Day 25. Save the concept examples in day26/scratch/ and run each with python3 <file>; the Concept 3 examples need fde_test.

```bash
# One-time setup for Day 26, inside day26/ (macOS / Linux shell).
python3 -m venv .venv               # a fresh venv (Day 6)
source .venv/bin/activate           # Windows: .venv\Scripts\activate
python3 -m pip install "fastapi[standard]" "psycopg[binary,pool]" \
    pydantic-settings pytest
cp .env.example .env                # .env stays out of git (Day 24)
psql -d fde -v ON_ERROR_STOP=1 -f schema.sql   # tables for the server
python3 -m pytest -q
# -> 46 failed, 1 passed     every failure ends in NotImplementedError.
# The one pass is test_health: the given layers already work.
# Tonight's target: the same command ends in "47 passed".
```

## Recap (2 minutes, from memory)

- Why must a test of this API use with TestClient(app) as client, and what error do you get without the with?

- Day 25's conftest points the app at fde_test with monkeypatch.setenv and get_settings.cache_clear(), not with dependency_overrides on get_settings. Why?

- A handler registered for FastAPI's HTTPException does not change the 404 for an unknown URL. Why not, and what is the fix?

## Concept 1: Webhooks, the CMS calls you

### Why it matters for an FDE

So far every request in your code started on your side: you called an API (Day 6) or a client called yours (Days 23 to 25). Enterprise integrations need the opposite direction too. When an editor publishes a page in Contentstack, the search index, the CDN, the mobile app and now your AI assistant must know within seconds. Polling the CMS every minute is slow, costs API quota and misses deletes. A webhook is the CMS calling a URL you own the moment something happens.

You already know webhooks from the CMS side: in your Contentstack years you configured them to trigger site rebuilds. As an FDE you build the receiving end, and in almost every customer project at least one system (a CMS, Salesforce, Stripe, Slack, GitHub, a ticketing tool) talks to your service this way.

### How it works

A webhook is an ordinary HTTP POST with a JSON body, sent by the CMS to a URL you register (here /webhooks/cms). Contentstack's payload has the event name, when it was triggered, and the entry itself under event_data. Your endpoint reads it, does its work and answers. The answer matters: Contentstack counts a 2xx as delivered, and a non-2xx or no answer within its timeout as failed, and then retries (the retry count and delay are configured per webhook).

One detail changes how you write the route. Until today you declared a Pydantic model as the body and let FastAPI parse it. For a signed webhook you need the raw bytes first, because the signature (Concept 2) is computed over the exact bytes that were sent. await request.body() gives them. React analogy: it is the difference between reading event.target.value and the already transformed state; you want what really arrived.

In [ ]:
# A webhook is just a POST that the CMS sends to YOUR URL.
import json
from fastapi import FastAPI, Request
from fastapi.testclient import TestClient

app = FastAPI()
received: list[bytes] = []  # stands in for "what reached our server"


@app.post("/webhooks/cms")
async def receive(request: Request) -> dict:
    # await request.body(): the RAW bytes exactly as they arrived.
    # Like reading req.body in Express BEFORE express.json() parses it.
    raw = await request.body()
    received.append(raw)
    payload = json.loads(raw)  # bytes -> dict, like JSON.parse()
    return {"status": "received", "event": payload["event"]}


client = TestClient(app)
event = {"event": "content_types.entries.publish",
         "event_data": {"entry": {"uid": "blt1", "title": "Home"}}}

# The CMS serialises the dict to bytes and POSTs them.
resp = client.post("/webhooks/cms", content=json.dumps(event),
                   headers={"Content-Type": "application/json"})
print(resp.status_code, resp.json())
# -> 200 {'status': 'received', 'event': 'content_types.entries.publish'}

# The SAME data, serialised another way, is DIFFERENT bytes.
compact = json.dumps(event, separators=(",", ":"))
print(len(json.dumps(event)), len(compact))  # -> 101 94
print(json.loads(compact) == event)  # -> True   same data...
print(compact.encode() == received[0])  # -> False  ...different bytes
# GOTCHA: a signature is computed over BYTES. Re-serialising the parsed
# dict gives other bytes, so always verify the raw body (Concept 2).

Retries have a consequence that surprises everyone the first time. If your endpoint does the work but answers too slowly, the CMS sees a timeout and sends the same event again. Delivery is at least once: the same event can arrive twice, and two events can arrive in the wrong order, because retries and parallel deliveries overtake each other.

In [ ]:
# Why the same event arrives twice: the sender retries until it sees a 2xx.
import time


def flaky_receiver(attempt: int, seen: list[str], event_id: str) -> int:
    seen.append(event_id)  # the work was DONE on every attempt...
    if attempt == 1:
        time.sleep(0.02)  # ...but attempt 1 was too slow to answer
        return 504  # the sender gave up waiting (a timeout)
    return 200


def cms_send(event_id: str, seen: list[str], max_tries: int = 3) -> None:
    # Roughly what Contentstack does: POST, and on a non-2xx or a timeout,
    # wait and try again (the retry count and delay are configurable).
    for attempt in range(1, max_tries + 1):
        status = flaky_receiver(attempt, seen, event_id)
        print(f"attempt {attempt}: {status}")
        if 200 <= status < 300:  # only a 2xx counts as "delivered"
            return


seen: list[str] = []
cms_send("publish:blt1:v7", seen)
# -> attempt 1: 504
# -> attempt 2: 200
print(seen)
# -> ['publish:blt1:v7', 'publish:blt1:v7']
# The receiver processed ONE event TWICE. That is "at-least-once" delivery.
# REMEMBER: webhooks arrive at least once, maybe late, maybe out of order.

### Real-world FDE example

A retail customer's product pages are in Contentstack, and their AI shopping assistant answers from your copy of the catalogue in Postgres. Merchandisers complained that the assistant quoted last week's prices. The fix was not a faster nightly sync: you register a webhook for publish, unpublish and delete, and the assistant's data is current within seconds of a publish. That is exactly today's task, and the same receiver later feeds a RAG index (Week 7).

### Common mistakes

- Declaring a Pydantic body parameter on a signed webhook. FastAPI parses and validates first, so a forged request reaches your validation code before any signature check, and the raw bytes are harder to get.

- Doing all the slow work before answering. The CMS times out, retries, and you process the event twice (Concept 5 fixes this).

- Assuming one delivery per event, in order. Code that is correct only for one ordered delivery corrupts data the first time a retry happens (Concept 3).

- Answering 500 for an event type you do not handle. The CMS retries it forever. Answer 200 with "ignored" for events you do not care about.

## Concept 2: HMAC signatures, proving who sent it

### Why it matters for an FDE

Your webhook URL is on the public internet. Anyone who learns it can POST a fake "publish" that puts a scam link on the customer's home page, or a fake "delete" that empties their catalogue. The customer's security team will ask one question in the review: "how do you know a request came from our CMS?" The standard answer, used by Contentstack, Stripe, GitHub, Shopify and Slack, is an HMAC signature.

### How it works

A plain hash (SHA-256) turns any bytes into a fixed 64-character fingerprint, but anyone can compute it, so it proves nothing about the sender. HMAC mixes a shared secret into the hash. The CMS and your service both know the secret; nobody else does. The CMS computes HMAC(secret, message) and sends it in a header; you compute the same thing over the bytes you received and compare. A match proves two things: the sender knew the secret, and not one byte changed on the way.

Contentstack's HMAC mode sends x-contentstack-hmac-signature: t=<unix time>,v1=<hex> and signs the string <t>.<raw body>. Putting the timestamp inside the signed bytes stops a replay: an attacker who captured a real request cannot resend it tomorrow, because you reject timestamps older than a few minutes and they cannot change t without breaking the signature. Note that Contentstack's default signing method is a different one (an RSA signature checked with a public key); the HMAC mode is enabled per organisation under Administration, Security Configuration, HMAC Signing. Check which one a customer uses on day one.

In [ ]:
# HMAC: prove that a body came from someone who knows the shared secret.
import hashlib
import hmac

SECRET = "whsec_demo_only"  # FDE TIP: in real code this comes from settings
body = b'{"event":"content_types.entries.publish","uid":"blt1"}'

# 1. A plain hash is NOT a signature: anyone can compute it.
print(hashlib.sha256(body).hexdigest()[:16])  # -> 70d33a20ccca93b0

# 2. HMAC mixes the secret into the hash. Without the secret, nobody can
#    produce the right value. Like crypto.createHmac('sha256', key) in Node.
def compute_signature(secret: str, timestamp: int, body: bytes) -> str:
    # Contentstack's HMAC mode signs "<timestamp>.<raw body>".
    signed = f"{timestamp}.".encode() + body  # bytes + bytes
    return hmac.new(secret.encode(), signed, hashlib.sha256).hexdigest()

t = 1795054500  # a Unix timestamp: seconds since 1 Jan 1970 (UTC)
sig = compute_signature(SECRET, t, body)
header = f"t={t},v1={sig}"  # the shape of x-contentstack-hmac-signature
print(header[:28] + "...")  # -> t=1795054500,v1=bf884d4c9e40...

# 3. The receiver recomputes and compares in CONSTANT time.
def verify(secret: str, header: str, body: bytes, now: int,
           tolerance_s: int = 300) -> bool:
    parts = dict(p.split("=", 1) for p in header.split(","))
    ts = int(parts["t"])
    if abs(now - ts) > tolerance_s:  # too old (or from the future)
        return False
    expected = compute_signature(secret, ts, body)
    # GOTCHA: never use == here. compare_digest takes the same time
    # whether the first or the last character differs, so an attacker
    # cannot guess the signature one character at a time.
    return hmac.compare_digest(expected, parts["v1"])

print(verify(SECRET, header, body, now=t + 10))  # -> True
print(verify(SECRET, header, body + b" ", now=t + 10))
# -> False   one extra space changed the bytes
print(verify("wrong-secret", header, body, now=t + 10))  # -> False
print(verify(SECRET, header, body, now=t + 3600))
# -> False   a replay: a real request captured an hour ago
# REMEMBER: sign the raw bytes, check the timestamp, compare_digest.

Two details are easy to get wrong. First, compare with hmac.compare_digest, never ==: == stops at the first wrong character, so its timing tells an attacker how many leading characters were right. Second, secrets get rotated. During a rotation Contentstack signs with both keys and sends two v1 values; you accept the request if any one matches.

In [ ]:
# Key rotation: during a switch, the header carries one v1 per secret.
import hashlib
import hmac

def sign(secret: str, t: int, body: bytes) -> str:
    return hmac.new(secret.encode(), f"{t}.".encode() + body,
                    hashlib.sha256).hexdigest()

body, t = b'{"uid":"blt1"}', 1795054500
# While Contentstack rotates the org key, BOTH keys sign the payload.
header = f"t={t},v1={sign('new-key', t, body)},v1={sign('old-key', t, body)}"

# GOTCHA: dict(...) keeps only the LAST v1. Collect every v1 into a list.
pairs = [part.split("=", 1) for part in header.split(",")]
print(len(dict(pairs)))  # -> 2   keys 't' and 'v1': one signature was lost
v1s = [value for key, value in pairs if key == "v1"]
print(len(v1s))  # -> 2

# Our service still has only the OLD secret configured: accept any match.
expected = sign("old-key", t, body)
print(any(hmac.compare_digest(expected, v) for v in v1s))  # -> True
# FDE TIP: this is what lets the customer rotate a secret with zero downtime.

### Real-world FDE example

On a bank's knowledge-assistant project, the pen test team sent your webhook a captured, validly signed request two hours after it was recorded. Because the timestamp was signed and you rejected anything older than five minutes, it got a 401 and the finding closed in a day. The team that had signed only the body (no timestamp) had to redesign during the security review, which delayed their go-live by two weeks.

### Common mistakes

- Verifying json.dumps(parsed_body) instead of the raw bytes. Key order and spacing change, and valid requests fail at random.

- Comparing with ==. It works in tests and leaks timing in production.

- Returning different messages for "bad signature" and "too old". Every hint helps an attacker; one "invalid signature" is enough, and your logs can hold the detail.

- Putting the secret in code or in the repo. It belongs in settings (SecretStr, Day 24) and the customer's secret store.

- Forgetting rotation. If you accept exactly one v1, a key change in the CMS takes your integration down until you redeploy.

## Concept 3: Idempotency, the same event twice is safe

### Why it matters for an FDE

Concept 1 showed that a webhook can arrive twice and out of order. An operation is idempotent when doing it twice has the same effect as doing it once (Day 22: PUT and DELETE are, POST is not). Your receiver must be idempotent, or a retry sends two emails, charges twice, or puts an old title back on the live site. "What happens if this webhook is delivered twice?" is a standard system-design interview question for FDE and solutions roles.

### How it works

Two tools, both enforced by Postgres so that two deliveries arriving at the same moment cannot both win. Deduplication: a webhook_deliveries table whose primary key is an event key. Contentstack sends no delivery id, so you build one from what identifies a change: event, entry uid, locale and the entry's updated_at. INSERT ... ON CONFLICT DO NOTHING RETURNING tells you in one statement whether this event is new. A version guard: the upsert (Day 19) gets a WHERE on its UPDATE part, so a late, older version never overwrites a newer one.

In [ ]:
# Idempotency: process each event once, and never let an old one win.
import psycopg

URL = "postgresql:///fde_test"  # the TEST database (Day 25), never fde

with psycopg.connect(URL, autocommit=True) as conn:
    conn.execute("SET client_min_messages = warning")  # hide NOTICEs
    conn.execute("DROP TABLE IF EXISTS demo_deliveries, demo_entries")
    # One row per event we have already handled. The PRIMARY KEY is the
    # whole trick: Postgres itself refuses a second row with the same key.
    conn.execute("CREATE TABLE demo_deliveries (event_key text PRIMARY KEY)")
    conn.execute("""CREATE TABLE demo_entries (
        uid text PRIMARY KEY, title text NOT NULL,
        updated_at timestamptz NOT NULL)""")

    def first_time(key: str) -> bool:
        # ON CONFLICT DO NOTHING: a duplicate is not an error, it is a no-op.
        # RETURNING gives a row only when the INSERT really happened.
        row = conn.execute(
            "INSERT INTO demo_deliveries VALUES (%s)"
            " ON CONFLICT DO NOTHING RETURNING event_key", (key,),
        ).fetchone()
        return row is not None  # None means "seen before"

    print(first_time("publish:blt1:08:00"))  # -> True   new: do the work
    print(first_time("publish:blt1:08:00"))  # -> False  a retry: skip it

    def upsert(uid: str, title: str, updated_at: str) -> int:
        # Day 19's upsert, plus a WHERE on the UPDATE part: only overwrite
        # when the incoming version is NEWER than the stored one.
        cur = conn.execute(
            """INSERT INTO demo_entries VALUES (%s, %s, %s)
               ON CONFLICT (uid) DO UPDATE
               SET title = EXCLUDED.title, updated_at = EXCLUDED.updated_at
               WHERE demo_entries.updated_at < EXCLUDED.updated_at""",
            (uid, title, updated_at),
        )
        return cur.rowcount  # 1 = written, 0 = skipped as stale

    # Deliveries arrive OUT OF ORDER: version 2 first, then version 1.
    print(upsert("blt1", "Home v2", "2026-11-19 08:05+00"))  # -> 1
    print(upsert("blt1", "Home v1", "2026-11-19 08:00+00"))  # -> 0
    print(conn.execute("SELECT title FROM demo_entries").fetchone())
    # -> ('Home v2',)   the late, older event did not overwrite the newer one
    # EXCLUDED is the row you TRIED to insert (Day 19).
# REMEMBER: dedupe by a key, and compare versions before you overwrite.

The third rule is about transactions (Day 18). Recording the delivery and applying it must commit together. If they are separate commits and the process dies between them, the event is marked as seen but was never applied, and every retry is skipped. Day 24's get_conn already runs the whole request in one transaction, so in the route this is free, as long as you do not commit by hand. Run this example after the previous one (it reuses its tables).

In [ ]:
# Record the delivery and apply it in ONE transaction, or a crash loses data.
import psycopg

URL = "postgresql:///fde_test"

with psycopg.connect(URL) as conn:  # not autocommit: we control commits
    conn.execute("TRUNCATE demo_deliveries, demo_entries")
    conn.commit()

    def handle(key: str, title: str, crash: bool) -> None:
        try:
            with conn.transaction():  # BEGIN ... COMMIT, or ROLLBACK on error
                conn.execute("INSERT INTO demo_deliveries VALUES (%s)"
                             " ON CONFLICT DO NOTHING", (key,))
                if crash:
                    raise RuntimeError("pod killed mid-request")
                conn.execute("INSERT INTO demo_entries VALUES"
                             " ('blt9', %s, now())", (title,))
            print("committed")
        except RuntimeError as exc:
            print("rolled back:", exc)

    handle("publish:blt9:v1", "Pricing", crash=True)
    # -> rolled back: pod killed mid-request
    seen = conn.execute("SELECT count(*) FROM demo_deliveries").fetchone()
    print(seen)  # -> (0,)   the delivery record was rolled back too...
    handle("publish:blt9:v1", "Pricing", crash=False)  # ...so the retry works
    # -> committed
    print(conn.execute("SELECT title FROM demo_entries").fetchone())
    # -> ('Pricing',)
    # GOTCHA: with two separate commits, the delivery would be marked "seen"
    # but the entry never written, and every retry would be skipped forever.

### Real-world FDE example

A media customer's Contentstack sends about 40,000 webhooks on a busy election night, and their old receiver occasionally showed yesterday's headline after a fast correction: two deliveries raced and the older one committed last. The fix was this concept's version guard, one WHERE clause. You would find this in a discovery call by asking: "have you ever seen an old version come back after a quick re-publish?"

### Common mistakes

- Checking with SELECT first, then inserting. Two deliveries can both see "not there yet" and both process. Let the unique key decide.

- A dedupe key without the version (just the uid). The second, real publish of the same entry is then skipped as a "duplicate".

- Using the time you received the event as the version. Arrival order is exactly what you cannot trust; use the CMS's own updated_at.

- Committing the delivery record in its own transaction before the work (the GOTCHA above).

## Concept 4: API keys, who is calling your API

### Why it matters for an FDE

Your sync service also has a read side: the customer's search team and their agency read synced entries from /v1/entries. Before full OAuth2 and SSO (Week 10), the most common way services authenticate to each other is an API key: a long random secret sent in a header. Contentstack's own Delivery API works this way (the stack API key plus a delivery token in headers), so you have been on the client side of this for years.

### How it works

Four rules make an API key setup professional. Generate keys with secrets, never random. Store only a hash of each key, like a password: a leaked .env or database then gives nobody a working key. Allow several hashes at once, so a client can move to a new key before the old one is removed (rotation). And check the key in a dependency, so each protected route says so in its signature, and a forgotten check is visible in code review.

FastAPI's APIKeyHeader reads the header and adds the padlock in /docs. With auto_error=False it returns None for a missing header and your dependency chooses the response. The status code matters: 401 means "I do not know who you are" (no key, wrong key), 403 means "I know you, and you may not do this" (a valid key without permission). Version note: since FastAPI 0.122 the built-in error for a missing key is 401; older tutorials show 403.

In [ ]:
# API keys: generate, store only a hash, check a header in a dependency.
import hashlib
import hmac
import secrets
from typing import Annotated

from fastapi import Depends, FastAPI, HTTPException, Security
from fastapi.security import APIKeyHeader
from fastapi.testclient import TestClient

# 1. Generate: secrets, not random. random is predictable; secrets is not.
key = "acme_" + secrets.token_urlsafe(24)  # like crypto.randomBytes in Node
print(key[:5], len(key))  # -> acme_ 37

# 2. Store only the SHA-256 hash, as you would a password digest. A leaked
#    database or .env then does not leak a usable key.
def hash_api_key(key: str) -> str:
    return hashlib.sha256(key.encode()).hexdigest()

DEMO_KEY = "acme_demo_key_123"
STORED = [hash_api_key(DEMO_KEY)]  # FDE TIP: in real code, from settings

# 3. APIKeyHeader reads X-API-Key and shows a padlock in /docs.
#    auto_error=False: return None when it is missing, so WE choose the error.
api_key_header = APIKeyHeader(name="X-API-Key", auto_error=False)

def require_api_key(
    key: Annotated[str | None, Security(api_key_header)],
) -> str:
    if key is not None:
        digest = hash_api_key(key)
        for stored in STORED:  # several hashes = rotation with no downtime
            if hmac.compare_digest(digest, stored):
                return stored[:8]  # a short key id that is safe to log
    # 401 = "who are you?" (no or bad credentials).
    # 403 = "I know you, but you may not do this" (Week 10, roles).
    raise HTTPException(status_code=401, detail="invalid or missing API key",
                        headers={"WWW-Authenticate": "APIKey"})

app = FastAPI()

@app.get("/v1/entries/{uid}")
def get_entry(uid: str,
              key_id: Annotated[str, Depends(require_api_key)]) -> dict:
    return {"uid": uid, "key_id": key_id}

client = TestClient(app)
print(client.get("/v1/entries/blt1").status_code)  # -> 401
bad = client.get("/v1/entries/blt1", headers={"X-API-Key": "guess"})
print(bad.status_code, bad.json())
# -> 401 {'detail': 'invalid or missing API key'}
ok = client.get("/v1/entries/blt1", headers={"x-api-key": DEMO_KEY})
print(ok.status_code, ok.json() == {"uid": "blt1", "key_id": STORED[0][:8]})
# -> 200 True   header names are case-insensitive in HTTP
# GOTCHA: never put the key in the URL (?api_key=...). URLs end up in logs,
# browser history and proxies. Headers are the place for credentials.

### Real-world FDE example

An insurance customer gives three vendors access to your claims summary API. Each vendor gets its own key, and your logs record the short key id with every request. When one vendor's laptop is stolen, you remove one hash from the settings and redeploy; the other two vendors notice nothing, and the audit log shows exactly what the stolen key read. One shared key for everyone would have meant rotating all three vendors at once, on a Friday night.

### Common mistakes

- Keys in the URL (?api_key=). URLs are written to access logs, proxies and browser history.

- Storing or logging the raw key. Log the short hash id instead.

- One key shared by every client. You cannot revoke one client or tell them apart in logs.

- Answering 403 for a missing key, or 404 to "hide" the endpoint. Clients and their retry logic rely on 401 meaning "fix your credentials".

- Using an API key where a user is involved. Keys identify systems; people log in with SSO and OAuth2 (Week 10).

## Concept 5: Background tasks, answer fast and finish later

### Why it matters for an FDE

After a publish you often have follow-up work: reindex search, purge the CDN, re-embed the page for the RAG index (Week 7). That work is slow and talks to other services. If the webhook waits for it, the CMS times out and retries (Concept 1). The rule is: do the essential part (the database write) inside the request, answer, and hand the extras to something that runs afterwards.

### How it works

FastAPI injects a BackgroundTasks object when a route asks for one, like it injects Request. add_task(function, *args) queues a call that runs after the response has been sent, in the same process. JavaScript analogy: it is close to firing a promise without awaiting it after res.json() in Express, but managed and run in order. In tests, TestClient waits for the tasks before it returns, which makes them easy to assert on.

In [ ]:
# BackgroundTasks: answer the CMS fast, do the slow follow-up afterwards.
import time

from fastapi import BackgroundTasks, FastAPI
from fastapi.testclient import TestClient

app = FastAPI()
events: list[str] = []  # an ordered log of what happened


def reindex_search(uid: str) -> None:
    # Stands in for slow work: refresh a search index, purge a CDN cache.
    time.sleep(0.2)
    events.append(f"reindexed {uid}")


def broken_task(uid: str) -> None:
    raise ValueError(f"search service down for {uid}")


@app.post("/webhooks/cms")
def receive(background_tasks: BackgroundTasks) -> dict:
    # FastAPI injects BackgroundTasks for you, like Request.
    events.append("saved to db")  # the important part, done NOW
    # add_task(function, *args): run it AFTER the response is sent.
    background_tasks.add_task(reindex_search, "blt1")
    events.append("response built")
    return {"status": "processed"}


@app.post("/webhooks/broken")
def receive_broken(background_tasks: BackgroundTasks) -> dict:
    background_tasks.add_task(broken_task, "blt2")
    return {"status": "processed"}


client = TestClient(app)
start = time.perf_counter()
resp = client.post("/webhooks/cms")
print(resp.json(), events)
# -> {'status': 'processed'} ['saved to db', 'response built',
#    'reindexed blt1']
# GOTCHA: TestClient waits for background tasks before it returns, so the
# test sees 'reindexed' already. A real client gets the response FIRST.
print(round(time.perf_counter() - start, 1))  # -> 0.2   (TestClient waited)

safe = TestClient(app, raise_server_exceptions=False)
print(safe.post("/webhooks/broken").status_code)
# -> 200   the response was already sent when the task failed
# The CMS thinks all is well and will NOT retry. A background task has no
# retry and is lost if the process dies: fine for a cache purge, not for
# the database write. REMEMBER: write first, then hand off the extras.

The limits are as important as the feature. A background task has no retry and lives in memory, so it is lost if the process restarts, and an error in it is only logged: the client already has its 200. Use it for work you can afford to lose or redo (a cache purge, a reindex that the next publish repeats). For work that must happen (sending an invoice, writing the entry) use the request itself, or a real queue with retries (SQS, Pub/Sub, Celery) which you will meet in Phase 3.

### Real-world FDE example

A travel customer's webhook endpoint called their search vendor inline and took 6 to 9 seconds; Contentstack's retries then doubled their load during sales. You moved the search call into a background task: the webhook now answers in about 40 ms, retries stopped, and if a reindex fails, the next publish of that page fixes it. Later, when reindex failures had to be reported, the same function moved onto a queue unchanged.

### Common mistakes

- Writing to the database in a background task. If it fails, the CMS already got its 200 and will never retry, so the change is lost.

- Passing the request's database connection into the task. The connection goes back to the pool when the request ends; open a new one inside the task if it needs the database.

- Long CPU work in a task. It still runs in your server process and slows other requests; that belongs in a worker.

- Testing only with TestClient and assuming the client waits. Real clients get the response first; never return data the task has not produced yet.

## Check your understanding (predict first, then run)

Save the file below as day26/scratch/predict.py. Before running it, write down what each of the six lines will print, and why. Then run it with python3 predict.py and explain any surprise in one sentence.

In [ ]:
# Six predictions. Write each answer down BEFORE you run the file.
import hashlib
import hmac
import json
from typing import Annotated

from fastapi import BackgroundTasks, FastAPI, Security
from fastapi.security import APIKeyHeader
from fastapi.testclient import TestClient

def sig(secret: bytes, body: bytes) -> str:
    return hmac.new(secret, body, hashlib.sha256).hexdigest()

a = json.dumps({"uid": "blt1", "v": 2}).encode()
b = json.dumps({"v": 2, "uid": "blt1"}).encode()
print("Q1", json.loads(a) == json.loads(b), sig(b"k", a) == sig(b"k", b))
print("Q2", len(sig(b"k", a)), len(sig(b"a much longer secret", a)))
print("Q3", hashlib.sha256(a).hexdigest() == sig(b"", a))

app = FastAPI()
log: list[str] = []

@app.get("/strict")
def strict(k: Annotated[str, Security(APIKeyHeader(name="X-API-Key"))]):
    return {"k": k}

@app.post("/hook")
def hook(tasks: BackgroundTasks) -> dict:
    tasks.add_task(log.append, "task")
    log.append("route")
    return {"log": list(log)}

client = TestClient(app)
print("Q4", client.get("/strict").status_code)
print("Q5", client.post("/hook").json(), log)
pairs = [p.split("=", 1) for p in "t=1,v1=aa,v1=bb".split(",")]
print("Q6", dict(pairs))

## Hands-on task (about 2 hours): Acme's content sync service

Scenario: Acme Retail wants its AI shopping assistant to answer from content that is never more than a few seconds old. Their Contentstack will send publish, unpublish and delete webhooks to your service, signed in HMAC mode, and their search team will read the synced entries with API keys. Their security lead has three conditions before go-live: forged and replayed requests are rejected, retries never duplicate or roll back data, and keys can be rotated without downtime.

| File | Given | You write | Run while working |
|---|---|---|---|
| signatures.py | docstrings, examples | 3 functions | pytest -q -k sig |
| api_keys.py | api_key_header | 3 functions | pytest -q -k key |
| repository.py | get_entry() | event_key and 3 methods | pytest -q -k repo |
| sync_api.py | app, lifespan, health, read route | the webhook route body | pytest -q -k "webhook or read" |
| test_mine.py | 3 named stubs | 3 tests | pytest -q |

Also given, do not edit: settings.py, db.py, models.py, schema.sql, conftest.py, test_sync_api.py, send_webhook.py (a fake Contentstack that signs and sends a sample file) and two payloads in samples/. If your Day 24 db.py and Day 25 conftest.py are not finished, finish them before reading these two files.

- Step 0 (10 min). Do the Setup above. python3 -m pytest -q must show 46 failed, 1 passed, with no import errors. Read models.py and samples/publish_home.json side by side.

- Step 1: signatures.py (30 min). compute_signature, then parse_signature_header, then verify_signature. Rerun python3 -m pytest -q -k sig after each one; then python3 signatures.py must print the four expected lines.

- Step 2: api_keys.py (15 min). Three short functions. Then make your own local key: run python3 -c "import secrets; print('acme_' + secrets.token_urlsafe(24))", hash it with your hash_api_key, and put the hash (never the key) in ACME_API_KEY_HASHES in .env.

- Step 3: repository.py (30 min). event_key, then record_delivery, upsert_entry and delete_entry. Run your SQL by hand in psql on fde_test first if a test fails (Day 24 habit).

- Step 4: the route (30 min). Follow the seven numbered lines of the contract above receive_webhook. When it is done, the full given suite shows 44 passed.

- Step 5: test_mine.py (10 min). Three tests, from the hints. Then python3 -m pytest -q shows 47 passed.

- Step 6: the manual run (5 min). Start fastapi dev sync_api.py. In a second terminal run python3 send_webhook.py samples/publish_home.json twice, then once with --secret wrong, once with --age 3600, then samples/unpublish_home.json. Finish with a curl GET of /v1/entries/blt_home_001 with your key, before and after the unpublish. Paste the commands and outputs into day26/delivery_log.txt.

- Step 7. Run ruff check . and mypy signatures.py api_keys.py repository.py, then commit and push.

Acceptance criteria:

- python3 -m pytest -q inside day26/ shows 47 passed. Do not edit the given files.

- delivery_log.txt shows processed, duplicate, 401 for the wrong secret, 401 for the old timestamp, processed for the unpublish, and the GET returning 200 and then 404.

- The signature is verified on the raw body before any JSON parsing, and every comparison of secrets uses hmac.compare_digest.

- No secret, key or full key hash appears in code, logs, responses or git; .env is not committed.

- Duplicates and stale versions are decided by Postgres (ON CONFLICT and the WHERE guard), with no SELECT-then-INSERT, and nothing in repository.py calls commit().

- Committed and pushed with a message such as "Day 26: signed CMS webhook receiver with idempotent upserts and API keys" (a WIP push counts).

Hints, not answers:

- If every webhook test gets 401 "invalid signature" but the sig tests pass, the route is verifying something other than the raw bytes, or reading the wrong setting. Print the first 20 bytes of what you verify.

- If the duplicate test shows "processed" twice, record_delivery is not returning False on a conflict: check RETURNING and fetchone().

- If the stale test fails, run your upsert twice in psql with an older updated_at and look at the row count Postgres prints (UPDATE 0 or INSERT 0 1).

- If the 422 tests get 500, the ValidationError escaped: catch pydantic's ValidationError around model_validate_json.

- test_mine.py test 3 raises inside the test because the client fixture re-raises server errors (Day 25, Concept 1). That is what pytest.raises is for.

Stretch (only if the task and the push are done): add POST /v1/admin/replay/{event_key} protected by a separate admin key, which deletes one row from webhook_deliveries so that support can force a re-sync of one event. Write two tests for it, including one proving the normal read key gets 403, not 401.

Catch-up (1h): first push whatever exists of Days 22 to 25 (15 min). Then finish Day 25 until its 37 tests pass, because Day 27 packages Days 24 to 26 into one project.

## DSA practice (30 min)

Both problems are in day26/dsa.py with examples and hints; python3 dsa.py prints PASS or FAIL for each example. Give each 15 minutes before reading the hints.

- constant_time_equals: compare two strings without stopping at the first difference, the idea behind hmac.compare_digest. Hint: XOR the character codes and OR them into one accumulator.

- apply_events: replay a stream of webhook events into a final state, skipping repeated keys and versions older than the stored one, and applying deletes. This is your receiver's logic as a pure function. Hint: a set for seen keys, a dict for the state.

## Quiz (3 quick questions)

- Why does the route read await request.body() instead of declaring a Pydantic model parameter, and what would break if it verified json.dumps(payload.model_dump()) instead?

- Your dedupe table records deliveries in its own transaction, committed before the upsert. Describe the failure this allows and how one transaction prevents it.

- A teammate moves the database upsert into a background task "so the webhook answers faster". What can now go wrong, and what would you move into the background instead?

## What you achieved today

You built the receiving end of an enterprise integration: a webhook that proves where each request came from, rejects replays, survives retries and out-of-order deliveries because Postgres enforces the rules, protects its read side with rotatable, hashed API keys, and keeps slow work out of the request. Together with Days 24 and 25, this is the shape of a service a customer's security team signs off on.

Tomorrow, Day 27: the Phase 1 milestone. Days 24 to 26 become one project, content-sync-api, with uv, logging, a README, OpenAPI docs and a 3-minute demo script.

## When you're done

Paste into the Forward Deployment Engineer project chat as Day 26 review: signatures.py, repository.py, the route body, test_mine.py, the last lines of your pytest output, delivery_log.txt, your six predictions next to the real results, your quiz answers, dsa.py, and the link to your latest commit. Say whether Days 22 to 25 are pushed.

## Recap answers

*Your answers:*
